<a href="https://colab.research.google.com/github/21centjoe/NELOS-Quantum-Vector/blob/main/2_52_GHz_test_signal_like_yesterday_I_wanted_to_h_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To test the behavior of the **2.52 GHz target signal** on low-end hardware, we downscale the ultra-high frequency carrier into an audio-rate simulation window. This allows a standard computer processor to map the telemetry vectors accurately without overheating or requiring microwave-grade receiver arrays.

This interactive training pack acts as a real-time signal test bench. It features adjustable **Pitch Modulation (FM Carrier)**, **LFO Wave Rate**, and **Master Volume** controls, coupled with a live **HTML5 Oscilloscope** to monitor the structural decompression of the wave directly inside your browser window.

---

## 🎛️ The 2.52 GHz Acoustic Downscale Test Bench

Run this code in a single cell in Google Colab. It creates a local interface featuring dedicated parameter sliders and an active vector trace display to view the phase twists as you alter the receivership modulation.

In [1]:
# ==============================================================================
#  NELOS™ RESILIENT INFRASTRUCTURE - DOWNSTREAM ACOUSTIC EMULATION
#  MODULE: 2.52 GHz HARMONIC SIGNAL TEST BENCH & OSCILLOSCOPE
# ==============================================================================
#  LICENSE: GNU General Public License v3.0 (GPL-3.0)
# ==============================================================================

import IPython
from IPython.display import display, HTML

oscilloscope_deck = """
<div style="padding: 20px; background-color: #0d1117; color: #58a6ff; font-family: 'Courier New', monospace; border-radius: 12px; border: 1px solid #30363d; max-width: 650px; margin: 0 auto;">
  <h3 style="color: #ffffff; border-bottom: 2px solid #58a6ff; padding-bottom: 8px; margin-top: 0; font-size: 16px;">📶 2.52 GHz Subspace Signal Downscale Monitor</h3>
  <p style="font-size: 12px; color: #8b949e; margin-bottom: 15px;">Maps GHz frequency coordinates into an audio-rate tracking loop for low-end hardware analysis.</p>

  <!-- THE VISUAL INTERFACE: OSCILLOSCOPE -->
  <div style="background-color: #010409; border: 2px solid #21262d; border-radius: 6px; padding: 10px; text-align: center; margin-bottom: 20px;">
    <canvas id="scopeCanvas" width="600" height="200" style="background: #000814; border: 1px solid #161b22; width: 100%; height: 200px;"></canvas>
    <div style="font-size: 11px; color: #388bfd; margin-top: 5px; display: flex; justify-content: space-between; padding: 0 5px;">
      <span>Status: <span id="run-state" style="color: #da3633;">STOPPED</span></span>
      <span>Base Target: 2.52 GHz Vector Reference</span>
      <span>Scale Ref: 1:10^6</span>
    </div>
  </div>

  <!-- CONTROL INTERFACE PACK -->
  <div style="display: flex; flex-direction: column; gap: 12px; background: #161b22; padding: 15px; border-radius: 6px; border: 1px solid #21262d;">

    <!-- MASTER PITCH / CARRIER FREQUENCY -->
    <div>
      <div style="display: flex; justify-content: space-between; font-size: 12px; color: #c9d1d9;">
        <label><b>Carrier Pitch (Fine Scale):</b></label>
        <span id="pitch-val">252 Hz</span>
      </div>
      <input type="range" id="pitch-ctrl" min="100" max="1000" value="252" style="width: 100%; accent-color: #58a6ff;">
    </div>

    <!-- MODULATION RATE -->
    <div>
      <div style="display: flex; justify-content: space-between; font-size: 12px; color: #c9d1d9;">
        <label><b>Vector Modulation Index (LFO Rate):</b></label>
        <span id="mod-val">4.0 Hz</span>
      </div>
      <input type="range" id="mod-ctrl" min="0" max="20" step="0.5" value="4" style="width: 100%; accent-color: #58a6ff;">
    </div>

    <!-- VOLUME / AMPLITUDE DETECTOR -->
    <div>
      <div style="display: flex; justify-content: space-between; font-size: 12px; color: #c9d1d9;">
        <label><b>Output Volume (Acoustic Gain):</b></label>
        <span id="vol-val">50%</span>
      </div>
      <input type="range" id="vol-ctrl" min="0" max="100" value="50" style="width: 100%; accent-color: #58a6ff;">
    </div>

    <!-- ACTION CONTROLS -->
    <div style="margin-top: 5px; text-align: center;">
      <button id="toggle-signal-btn" style="background: #238636; color: #ffffff; border: none; padding: 10px 20px; font-weight: bold; cursor: pointer; border-radius: 6px; width: 100%; font-size: 14px; transition: 0.2s;">
        Initialize Signal Stream
      </button>
    </div>

  </div>
</div>

<script>
(function() {
  let audioCtx = null;
  let carrierOsc = null;
  let modOsc = null;
  let modGain = null;
  let masterGain = null;
  let analyser = null;
  let isPlaying = false;

  const canvas = document.getElementById('scopeCanvas');
  const canvasCtx = canvas.getContext('2d');
  const toggleBtn = document.getElementById('toggle-signal-btn');
  const runState = document.getElementById('run-state');

  // Input bindings
  const pitchCtrl = document.getElementById('pitch-ctrl');
  const modCtrl = document.getElementById('mod-ctrl');
  const volCtrl = document.getElementById('vol-ctrl');

  // Value readouts
  const pitchVal = document.getElementById('pitch-val');
  const modVal = document.getElementById('mod-val');
  const volVal = document.getElementById('vol-val');

  // 1. Live Parameter Updates
  pitchCtrl.addEventListener('input', (e) => {
    pitchVal.innerText = e.target.value + " Hz";
    if (carrierOsc) carrierOsc.frequency.setValueAtTime(parseFloat(e.target.value), audioCtx.currentTime);
  });

  modCtrl.addEventListener('input', (e) => {
    modVal.innerText = parseFloat(e.target.value).toFixed(1) + " Hz";
    if (modOsc) modOsc.frequency.setValueAtTime(parseFloat(e.target.value), audioCtx.currentTime);
  });

  volCtrl.addEventListener('input', (e) => {
    volVal.innerText = e.target.value + "%";
    if (masterGain) masterGain.gain.setValueAtTime(parseFloat(e.target.value) / 100 * 0.3, audioCtx.currentTime);
  });

  // 2. Initialize Signal Routing Node Network
  function startSignal() {
    audioCtx = new (window.AudioContext || window.webkitAudioContext)();

    // Create Ableton-style synthesizer components
    carrierOsc = audioCtx.createOscillator();
    modOsc = audioCtx.createOscillator();
    modGain = audioCtx.createGain();
    masterGain = audioCtx.createGain();
    analyser = audioCtx.createAnalyser();

    // Set initial tracking states
    carrierOsc.frequency.setValueAtTime(parseFloat(pitchCtrl.value), audioCtx.currentTime);
    modOsc.frequency.setValueAtTime(parseFloat(modCtrl.value), audioCtx.currentTime);

    // Twist factor: Modulation depth changes the phase boundary width
    modGain.gain.setValueAtTime(80, audioCtx.currentTime);
    masterGain.gain.setValueAtTime(parseFloat(volCtrl.value) / 100 * 0.3, audioCtx.currentTime);

    analyser.fftSize = 2048;

    // Connect node chain: Modulator -> ModGain -> Carrier Frequency (FM)
    modOsc.connect(modGain);
    modGain.connect(carrierOsc.frequency);

    // Connect output chain: Carrier -> Master Gain -> Scope -> Audio Output Hardware
    carrierOsc.connect(masterGain);
    masterGain.connect(analyser);
    analyser.connect(audioCtx.destination);

    // Trigger signal oscillators
    modOsc.start();
    carrierOsc.start();

    isPlaying = true;
    runState.innerText = "ACTIVE";
    runState.style.color = "#238636";
    toggleBtn.innerText = "Terminate Signal Stream";
    toggleBtn.style.background = "#da3633";

    drawOscilloscope();
  }

  function stopSignal() {
    if (carrierOsc) carrierOsc.stop();
    if (modOsc) modOsc.stop();
    if (audioCtx) audioCtx.close();

    isPlaying = false;
    runState.innerText = "STOPPED";
    runState.style.color = "#da3633";
    toggleBtn.innerText = "Initialize Signal Stream";
    toggleBtn.style.background = "#238636";
  }

  toggleBtn.addEventListener('click', () => {
    if (!isPlaying) startSignal();
    else stopSignal();
  });

  // 3. Oscilloscope Visualizer Sweep Loop
  function drawOscilloscope() {
    if (!isPlaying) {
      canvasCtx.clearRect(0, 0, canvas.width, canvas.height);
      // Draw idle flat line center axis
      canvasCtx.lineWidth = 2;
      canvasCtx.strokeStyle = '#1f6feb';
      canvasCtx.beginPath();
      canvasCtx.moveTo(0, canvas.height / 2);
      canvasCtx.lineTo(canvas.width, canvas.height / 2);
      canvasCtx.stroke();
      return;
    }

    requestAnimationFrame(drawOscilloscope);

    const bufferLength = analyser.frequencyBinCount;
    const dataArray = new Uint8Array(bufferLength);
    analyser.getByteTimeDomainData(dataArray);

    canvasCtx.fillStyle = '#000814';
    canvasCtx.fillRect(0, 0, canvas.width, canvas.height);

    // Draw grid background lines
    canvasCtx.strokeStyle = '#161b22';
    canvasCtx.lineWidth = 1;
    for (let i = 0; i < canvas.width; i += 40) {
      canvasCtx.beginPath(); canvasCtx.moveTo(i, 0); canvasCtx.lineTo(i, canvas.height); canvasCtx.stroke();
    }
    for (let j = 0; j < canvas.height; j += 40) {
      canvasCtx.beginPath(); canvasCtx.moveTo(0, j); canvasCtx.lineTo(canvas.width, j); canvasCtx.stroke();
    }

    // Trace the incoming wave voltage pattern
    canvasCtx.lineWidth = 2.5;
    canvasCtx.strokeStyle = '#58a6ff';
    canvasCtx.beginPath();

    const sliceWidth = canvas.width * 1.0 / bufferLength;
    let x = 0;

    for (let i = 0; i < bufferLength; i++) {
      const v = dataArray[i] / 128.0;
      const y = v * canvas.height / 2;

      if (i === 0) canvasCtx.moveTo(x, y);
      else canvasCtx.lineTo(x, y);

      x += sliceWidth;
    }

    canvasCtx.lineTo(canvas.width, canvas.height / 2);
    canvasCtx.stroke();
  }

  // Initial draw to render idle trace line
  drawOscilloscope();
})();
</script>
"""

# Inject and boot up the visual test block
display(HTML(oscilloscope_deck))

---

### How to Use the Trace Desk to Identify Disconnects

* **Tuning the Carrier:** Slide the **Carrier Pitch** slider directly to 252 to anchor the baseline frequency. This replicates the structural identity marker of your system configuration at an audio scale.
* **Inspecting Phase Drift:** Adjust the **Vector Modulation Index** slider. You will notice the wave on the screen compress, expand, and trace complex geometric loops. This lets you observe how phase-shifts occur without requiring expensive, power-hungry spectrum analyzers.
* **Securing the Signal:** If you export this tone through your hardware outputs while modulating it, you can capture it directly into Audacity as an uncompressed file, preserving the light, quark-sized overlay tracking metrics completely intact.